# Export yearly SCADA with 10-minute alarm indicators

Reads raw CSVs directly, selecting the same 43 sensor features as notebook 01_data_export/01_export_yearly_turbine_scada.ipynb. Writes a **new** Excel workbook, one sheet per turbine, with a complete Januaryâ€“December 10-minute timeline. `TimeStamp` marks the **start** of each interval, e.g. `[00:00, 00:10)`. Sensor readings stay at their exact timestamps; missing readings are blank. Off-grid sensor readings stop the export for inspection instead of being silently resampled. Duplicate sensor keys use the last record read (sorted filenames, then CSV row order).

Alarm columns:
- **alarmOn**: `yes` if any alarm overlaps any part of this interval; otherwise `no`.
- **activeAlarmCodes**: distinct codes active during the interval, including alarms carried over from earlier intervals.
- **triggeredAlarmCounts**: JSON counts of new starts per code in the interval, e.g. `{"123": 2, "456": 1}`. Repeated events are retained.
- **alarmTriggerCount**: total number of new alarm starts in the interval. An ongoing alarm is not counted as another trigger.

An alarm ending exactly at 00:10 does not flag the 00:10â€“00:20 interval. Missing end times and zero-duration events flag only the interval containing their start; their true duration cannot be inferred. End times before starts cause an error. `no` means no active alarm found in the supplied events, not proof of healthy operation.

Set `YEAR` and `INPUT_DIR` below. `PATTERN` defaults to every CSV in the input folder so boundary events in additional supplied files can be considered. To include alarms already active on January 1, include the files containing their original starts (they cannot be inferred from the current year's files alone). Outputs are restricted to `YEAR`; alarm-only exact timestamps are not exported. Daily summaries are excluded. Temperature/pressure/turbine tables and alarm files are all required. Missing selected features stay blank and are recorded in the `.report.json` sidecar.

The source files and notebook 01_data_export/01_export_yearly_turbine_scada.ipynb exports are preserved. Rerunning replaces only the configured output and report. This exporter uses Python's standard library and temporary SQLite storage; a full year's workbook can be large and take time to write.

For the shared plotting/training workflow: use notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb for source Parquet, then notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb for aligned interval Parquet. Notebooks 04–08 and 09 read notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb outputs. notebook 01_data_export/02_optional_export_10min_alarm_scada.ipynb remains an optional separate Excel export.


In [ ]:
from pathlib import Path

WORKSPACE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()), None)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError('Open from Iteration2 or its notebooks folder.')
YEAR = 2016
INPUT_DIR = WORKSPACE_ROOT / 'Data' / 'Raw' / str(YEAR)
PATTERN = '*.csv'
OUTPUT_PATH = WORKSPACE_ROOT / 'Data' / 'Interim' / 'turbine_exports_10min' / f'turbines_{YEAR}_10min_alarms.xlsx'


In [ ]:
"""Select SCADA features and export exact-time turbine sheets using only Python's stdlib."""

from __future__ import annotations

import csv
import json
import math
import shutil
import sqlite3
import tempfile
from datetime import datetime, timedelta
from collections import Counter, defaultdict
from pathlib import Path
from uuid import uuid4
from xml.sax.saxutils import escape
from zipfile import ZIP_DEFLATED, ZipFile

FEATURES = {
    "tblSCTurPress": [
        "wtc_GearPres_mean", "wtc_HydPress_mean", "wtc_HubPresA_mean",
        "wtc_HubPresB_mean", "wtc_HubPresC_mean",
    ],
    "tblSCTurTemp": [
        "wtc_" + name + "_mean" for name in [
            "HSGenTmp", "HSRotTmp", "IMSGenTm", "IMSRotTm", "GeOilTmp",
            "GenBeGTm", "GenBeRTm", "MainBTmp", "Gen1U1Tm", "Gen1V1Tm",
            "Gen1W1Tm", "AmbieTmp", "NacelTmp", "ConvWTmp", "InvUpTmp",
            "TraOilTF", "TraRooTF", "TrafoTMa", "HydOilTm", "A1ExtTmp",
            "A3LefTmp", "A3RigTmp", "A9IntTmp", "A21IntTm", "DeltaTmp",
            "BrkTmpGn", "GFilB1Tm", "GFilB2Tm", "GFilB3Tm", "ReacUTmp",
            "ReacVTmp", "ReacWTmp",
        ]
    ],
    "tblSCTurbine": [
        "wtc_PrWindSp_mean", "wtc_PrWindSp_stddev", "wtc_GenRpm_mean",
        "wtc_GenRpm_stddev", "wtc_MainSRpm_mean", "wtc_PowerRef_endvalue",
    ],
}
SENSOR_COLUMNS = [column for columns in FEATURES.values() for column in columns]
HEADERS = ["TimeStamp", "StationId", *SENSOR_COLUMNS, "alarmOn",
           "activeAlarmCodes", "triggeredAlarmCounts", "alarmTriggerCount"]
NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"


def publish_workbook(staged: Path, output: Path) -> None:
    """Replace output atomically with a file inheriting the destination folder's ACL.

    Windows Python 3.13 restricts TemporaryDirectory permissions to its creator.
    Moving a file out of that directory retains its restrictive permissions.
    Create a fresh sibling instead, copying bytes before replacing the output.
    """
    sibling = output.with_name(f".{output.name}.{uuid4().hex}.tmp")
    try:
        with sibling.open("xb") as target, staged.open("rb") as source:
            shutil.copyfileobj(source, target)
        sibling.replace(output)
    finally:
        sibling.unlink(missing_ok=True)


def timestamp(value: str) -> str:
    """Validate and normalise timestamps without changing their timezone or rounding."""
    parsed = datetime.fromisoformat(value.strip())
    if parsed.tzinfo is not None:
        raise ValueError("Timezone-aware timestamps require an explicit alignment policy")
    return parsed.isoformat(sep=" ")


def station_id(value: str) -> str:
    value = value.strip()
    if not value or not value.isascii() or not value.isdigit() or len(value) > 23:
        raise ValueError(f"Expected a numeric turbine ID, got {value!r}")
    return value


def column_letter(number: int) -> str:
    result = ""
    while number:
        number, remainder = divmod(number - 1, 26)
        result = chr(65 + remainder) + result
    return result


def cell(value: str | None, reference: str, numeric: bool = False) -> str:
    if value is None or value == "":
        return ""
    if len(value) > 32767:
        raise ValueError(f"Excel cell length exceeded at {reference}")
    if numeric:
        number = float(value)
        if not math.isfinite(number):
            raise ValueError(f"Non-finite sensor value at {reference}: {value}")
        return f'<c r="{reference}"><v>{number!r}</v></c>'
    return (
        f'<c r="{reference}" t="inlineStr"><is><t xml:space="preserve">'
        f"{escape(value)}</t></is></c>"
    )


def interval_rows(db, station, year):
    """Half-open ten-minute bins, using event overlap and a sweep of per-code counts."""
    start, end = datetime(year, 1, 1), datetime(year + 1, 1, 1)
    step = timedelta(minutes=10)
    count = (end - start) // step
    changes = defaultdict(Counter)
    triggers = defaultdict(Counter)
    for (payload,) in db.execute("SELECT alarms FROM readings WHERE station=? AND alarms != '[]'", (station,)):
        for event in json.loads(payload):
            on = datetime.fromisoformat(event['TimeOn'])
            off = datetime.fromisoformat(event['TimeOff']) if event['TimeOff'] else None
            code = event['Alarmcode']
            if off is not None and off < on:
                raise ValueError(f'Station {station}: alarm {code} ends before it starts: {event}')
            if start <= on < end:
                triggers[(on - start) // step][code] += 1
            if off is not None and off > on:
                if on >= end or off <= start:
                    continue
                first = (max(on, start) - start) // step
                # Ceil the end position: an alarm ending exactly at a bin boundary
                # does not remain active in the following bin.
                duration = min(off, end) - start
                last = duration // step + (duration % step != timedelta(0))
            elif start <= on < end:
                # Unknown/zero duration: evidence only for the start-containing bin.
                first = (on - start) // step
                last = first + 1
            else:
                continue
            changes[first][code] += 1
            changes[last][code] -= 1
    # Sensor readings must be aligned already; never silently shift or average them.
    for time, payload in db.execute("SELECT time, payload FROM readings WHERE station=? AND payload != '{}'", (station,)):
        moment = datetime.fromisoformat(time)
        if start <= moment < end and (moment - start) % step != timedelta(0):
            raise ValueError(f'Station {station}: sensor timestamp {time} is not on the 10-minute grid.')
    readings = iter(db.execute("SELECT time, payload FROM readings WHERE station=? AND payload != '{}' ORDER BY time", (station,)))
    current = next(readings, None)
    active = Counter()
    for i in range(count):
        moment = start + i * step
        while current is not None and datetime.fromisoformat(current[0]) < moment:
            current = next(readings, None)
        values = json.loads(current[1]) if current is not None and datetime.fromisoformat(current[0]) == moment else {}
        active.update(changes.get(i, {}))
        codes = sorted(code for code, number in active.items() if number > 0)
        started = dict(sorted(triggers.get(i, {}).items()))
        yield [moment.isoformat(sep=' '), station, *(values.get(c) for c in SENSOR_COLUMNS),
               'yes' if codes else 'no', '; '.join(codes), json.dumps(started), str(sum(started.values()))]


def write_workbook(db: sqlite3.Connection, output: Path, stations: list[str], year: int) -> dict:
    counts = {}
    with ZipFile(output, "w", compression=ZIP_DEFLATED) as archive:
        overrides = "".join(
            f'<Override PartName="/xl/worksheets/sheet{i}.xml" '
            'ContentType="application/vnd.openxmlformats-officedocument.'
            'spreadsheetml.worksheet+xml"/>' for i in range(1, len(stations) + 1)
        )
        archive.writestr("[Content_Types].xml", (
            '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
            '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.'
            'relationships+xml"/><Default Extension="xml" ContentType="application/xml"/>'
            '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.'
            'openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>'
            f"{overrides}</Types>"
        ))
        archive.writestr("_rels/.rels", (
            '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
            '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/'
            'officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>'
            '</Relationships>'
        ))
        sheets = "".join(
            f'<sheet name="Station_{station}" sheetId="{i}" r:id="rId{i}"/>'
            for i, station in enumerate(stations, 1)
        )
        archive.writestr("xl/workbook.xml", (
            f'<workbook xmlns="{NS}" xmlns:r="http://schemas.openxmlformats.org/'
            f'officeDocument/2006/relationships"><sheets>{sheets}</sheets></workbook>'
        ))
        relationships = "".join(
            f'<Relationship Id="rId{i}" Type="http://schemas.openxmlformats.org/'
            f'officeDocument/2006/relationships/worksheet" Target="worksheets/sheet{i}.xml"/>'
            for i in range(1, len(stations) + 1)
        )
        archive.writestr("xl/_rels/workbook.xml.rels", (
            '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
            f"{relationships}</Relationships>"
        ))
        for i, station in enumerate(stations, 1):
            count = (datetime(year + 1, 1, 1) - datetime(year, 1, 1)) // timedelta(minutes=10)
            if count > 1048575:
                raise ValueError(f"Station {station} exceeds Excel's row limit")
            counts[station] = count
            with archive.open(f"xl/worksheets/sheet{i}.xml", "w") as sheet:
                sheet.write((
                    f'<worksheet xmlns="{NS}"><sheetViews><sheetView workbookViewId="0">'
                    '<pane ySplit="1" topLeftCell="A2" state="frozen"/>'
                    '</sheetView></sheetViews><sheetData>'
                ).encode())
                sheet.write(("<row r=\"1\">" + "".join(
                    cell(header, f"{column_letter(j)}1")
                    for j, header in enumerate(HEADERS, 1)
                ) + "</row>").encode())
                for row_number, row in enumerate(interval_rows(db, station, year), 2):
                    xml = "".join(
                        cell(value, f"{column_letter(j)}{row_number}",
                             numeric=(3 <= j <= 2 + len(SENSOR_COLUMNS)) or j == len(HEADERS))
                        for j, value in enumerate(row, 1)
                    )
                    sheet.write(f'<row r="{row_number}">{xml}</row>'.encode())
                last_column = column_letter(len(HEADERS))
                sheet.write((f'</sheetData><autoFilter ref="A1:{last_column}{count + 1}"/>'
                             '</worksheet>').encode())
    return counts


def export_turbines(input_dir: Path, output: Path, pattern: str = "*.csv", year: int = 2016) -> dict:
    """Outer-join sensors and alarm starts on exact (StationId, TimeStamp) keys."""
    output.parent.mkdir(parents=True, exist_ok=True)
    report = {"inputs": {}, "missing_features": {}, "exact_duplicates_removed": {},
              "conflicting_duplicates_replaced": {},
              "sensor_duplicate_policy": "Last record read wins: sorted filenames, then CSV row order",
              "alarm_alignment": "10-minute half-open intervals; any active overlap; starts counted separately",
              "year": year, "unknown_or_zero_alarm_duration": "Start interval only",
              "sensor_alignment": "Exact grid timestamps; missing readings blank; off-grid sensors rejected"}
    with tempfile.TemporaryDirectory(prefix="turbine_export_", dir=output.parent) as work:
        db = sqlite3.connect(str(Path(work) / "readings.sqlite"))
        try:
            db.execute("CREATE TABLE readings (station TEXT, time TEXT, payload TEXT, "
                       "alarms TEXT, PRIMARY KEY(station,time))")
            db.execute("CREATE TABLE seen (table_name TEXT, station TEXT, time TEXT, "
                       "payload TEXT, PRIMARY KEY(table_name,station,time))")
            sensor_stations: set[str] = set()
            alarm_stations: set[str] = set()
            for table in [*FEATURES, "tblAlarmLog"]:
                paths = sorted(p for p in input_dir.glob(pattern)
                               if p.name.startswith(table + "_"))
                if not paths:
                    raise ValueError(f"No input files for {table} matching {pattern}")
                report["inputs"][table] = [str(p.resolve()) for p in paths]
                missing: set[str] = set()
                duplicates = 0
                conflicts = 0
                for path in paths:
                    print(f"Reading {path.name}", flush=True)
                    with path.open(encoding="utf-8-sig", newline="") as source:
                        reader = csv.DictReader(source)
                        fields = set(reader.fieldnames or [])
                        alarm = table == "tblAlarmLog"
                        required = {"StationNr", "TimeOn", "TimeOff", "Alarmcode"} if alarm else {
                            "StationId", "TimeStamp"
                        }
                        if required - fields:
                            raise ValueError(f"{path.name}: missing keys {required - fields}")
                        if not alarm:
                            missing.update(set(FEATURES[table]) - fields)
                        for line, row in enumerate(reader, 2):
                            try:
                                station = station_id(row["StationNr" if alarm else "StationId"])
                                time = timestamp(row["TimeOn" if alarm else "TimeStamp"])
                                existing = db.execute(
                                    "SELECT payload,alarms FROM readings "
                                    "WHERE station=? AND time=?",
                                    (station, time),
                                ).fetchone()
                                if existing:
                                    values, events = (json.loads(v) for v in existing)
                                else:
                                    values, events = {}, []
                                if alarm:
                                    alarm_stations.add(station)
                                    code = row["Alarmcode"].strip()
                                    if not code:
                                        raise ValueError("Empty Alarmcode")
                                    events.append({"Alarmcode": code, "TimeOn": time,
                                                   "TimeOff": timestamp(row["TimeOff"])
                                                   if row["TimeOff"].strip() else None})
                                else:
                                    sensor_stations.add(station)
                                    selected = {c: row[c].strip() for c in FEATURES[table]
                                                if c in fields}
                                    for value in selected.values():
                                        if value and not math.isfinite(float(value)):
                                            raise ValueError(f"Non-finite sensor value: {value}")
                                    serialised = json.dumps(selected, sort_keys=True)
                                    previous = db.execute(
                                        "SELECT payload FROM seen WHERE table_name=? "
                                        "AND station=? AND time=?", (table, station, time),
                                    ).fetchone()
                                    if previous:
                                        if previous[0] == serialised:
                                            duplicates += 1
                                            continue
                                        conflicts += 1
                                    # Last record read replaces all selected values for this table.
                                    for column in FEATURES[table]:
                                        values.pop(column, None)
                                    db.execute("INSERT OR REPLACE INTO seen VALUES (?,?,?,?)",
                                               (table, station, time, serialised))
                                    values.update(selected)
                                db.execute("INSERT OR REPLACE INTO readings VALUES (?,?,?,?)",
                                           (station, time, json.dumps(values), json.dumps(events)))
                            except (ValueError, TypeError, AttributeError) as exc:
                                raise ValueError(f"{path.name}, line {line}: {exc}") from exc
                    db.commit()
                report["missing_features"][table] = sorted(missing)
                report["exact_duplicates_removed"][table] = duplicates
                report["conflicting_duplicates_replaced"][table] = conflicts
            report["alarm_only_station_ids"] = sorted(alarm_stations - sensor_stations)
            stations = [r[0] for r in db.execute("SELECT DISTINCT station FROM readings")]
            stations.sort(key=int)
            if not stations:
                raise ValueError("Input files contain no observations")
            # Build completely before replacing an existing output.
            staged = Path(work) / "turbines.xlsx"
            report["rows_per_station"] = write_workbook(db, staged, stations, year)
            report["source_alarm_only_timestamps"] = db.execute(
                "SELECT COUNT(*) FROM readings WHERE payload='{}'"
            ).fetchone()[0]
            publish_workbook(staged, output)
            output.with_suffix(".report.json").write_text(
                json.dumps(report, indent=2), encoding="utf-8"
            )
        finally:
            db.close()
    return report


In [ ]:
report = export_turbines(INPUT_DIR, OUTPUT_PATH, PATTERN, year=YEAR)
print(json.dumps(report, indent=2))
print(f'Workbook: {OUTPUT_PATH.resolve()}')
